# Prompt Engineering 101 - Part VI.
## Agents

----

### *Managing Digital Workers*

## 1. The Anatomy of an Agent
An Agent is an AI Loop with three components:
* **The Brain (LLM):** Decides what to do next based on the current state.
* **The Hands (Tools):** Functions it can call (Search, Email, Calculator).
* **The Memory (Context):** A record of what it has already tried so it doesn't get stuck in a loop.

## 2. ReAct (Reasoning + Acting)
The standard loop for agents is **OODA**:
1.  **Observe:** "I am in a locked room."
2.  **Orient (Reason):** "I need a key. I should look under the mat."
3.  **Decide:** "I will call the `look_under_mat` tool."
4.  **Act:** *Executes code.*

## 3. Human-in-the-Loop
Agents are risky. They can delete files or spend money.
* **The Safety Valve:** We insert a "Permission Check" before the Agent executes a sensitive tool. The Agent must ask the human: "I want to spend $500. Proceed?"

## 4. Multi-Agent Systems
Instead of one super-smart AI, we use specialized teams.
* **Router Agent:** The "Traffic Cop" that sends tasks to the right expert.
* **Worker Agent:** The specialist (e.g., Coder, Writer, Researcher).

----

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini + Game Engine)
# We are installing the Google GenAI SDK.

!pip install -q -U google-genai

from google.colab import userdata
from google import genai
from IPython.display import display, Markdown
import time

# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 3. Create Wrapper Class (Same as Mod 4)
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-lite-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e

try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.5-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")


# 4. The Text Adventure Engine (Hidden Complexity)
class TextAdventure:
    def __init__(self):
        self.state = "You are in a stone corridor. North is a locked door. South is a dark pit. There is a brass key on the floor."
        self.inventory = []
        self.log = []
        
    def step(self, action):
        action = action.lower().strip()
        observation = ""
        
        if "pick up" in action and "key" in action:
            if "key" not in self.inventory:
                self.inventory.append("key")
                observation = "You picked up the brass key."
            else:
                observation = "You already have the key."
        elif "open" in action and "door" in action:
            if "key" in self.inventory:
                observation = "VICTORY! The door opens and you escape!"
                return observation, True
            else:
                observation = "The door is locked."
        elif "look" in action:
            observation = self.state
        else:
            observation = "Nothing happens."
            
        return observation, False

----

### **Phase 1: The Agent Brain (ReAct)**

In [ ]:
# @title 🧠 Topic 1: The Inner Monologue (Thought)
# Concept: Agents must "Think" before they "Act". If they just act, they make mistakes.
# We force the AI to output "Thought: [Reasoning]" first.

prompt = """
SCENARIO: You are a robot waiter. A customer spills water.
TASK: Decide what to do.
FORMAT:
Thought: [Your internal reasoning]
Action: [What you actually do]
"""

print(model.generate_content(prompt).text)


In [ ]:
# @title 🗣️ Topic 2: Parsing Actions (The Hands)
# Concept: The AI outputs text ("I will pick up the cup").
# We need to turn that into code (`robot.pickup(cup)`).
# This is called "Parsing".

# Simulation:
ai_response = "Thought: The floor is wet.\nAction: fetch_mop"

def parse_action(text):
    for line in text.split('\n'):
        if "Action:" in line:
            return line.replace("Action:", "").strip()
    return None

action = parse_action(ai_response)
print(f"AI Output: {ai_response}")
print(f"Parsed Command: '{action}'")

In [ ]:
# @title 🔄 Topic 3: Self-Correction (Getting Unstuck)
# Concept: Agents get stuck in loops (e.g., trying to open a locked door 5 times).
# We add a "History" so the agent sees its past failures.

history = """
Turn 1: Action: open_door -> Observation: Locked.
Turn 2: Action: open_door -> Observation: Locked.
Turn 3: Action: open_door -> Observation: Locked.
"""

prompt = f"""
HISTORY:
{history}

Current Observation: Locked.
Thought: I have tried this 3 times and it failed. I must try something else.
Action:
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🎮 LAB 1: The Dungeon Crawler
# TASK: Write a "System Prompt" that teaches the AI how to play the game.
# It needs to know:
# 1. Look around first.
# 2. Pick up items if seen.
# 3. Don't repeat failed actions.

# --- STUDENT WORKSPACE ---
system_prompt = """
You are a Dungeon Crawler Agent.
STRATEGY:
1. Always 'look' first to see what is in the room.
2. If you see a key, 'pick up' the key.
3. Once you have the key, 'open' the door.
"""
# -------------------------

# Run the Simulation
game = TextAdventure()
print(f"START: {game.state}")

history = ""
for i in range(5):
    # Construct prompt with history
    full_prompt = f"{system_prompt}\nHISTORY:\n{history}\nCURRENT STATE: {game.state}\nYour Move (Thought + Action):"
    
    # Agent decides
    response = model.generate_content(full_prompt).text
    print(f"\n--- Turn {i+1} ---")
    print(response)
    
    # Parse and Act
    action = parse_action(response)
    if not action: action = "look" # Fallback
    
    obs, win = game.step(action)
    print(f"🌍 World: {obs}")
    
    # Update History
    history += f"Action: {action} -> Observation: {obs}\n"
    
    if win: break

---

### **Phase 2: Tools & Safety**

In [ ]:
# @title 🧰 Topic 4: Function Calling (Tools)
# Concept: We give the AI a "Menu" of Python functions.

def calculate_tax(amount):
    return amount * 0.2

# We describe the tool to the AI (Mocking the API schema)
tool_definition = """
Tool: calculate_tax
Arguments: amount (integer)
Description: Calculates 20% tax on a value.
"""

prompt = f"""
{tool_definition}
User: How much tax on $100?
Output the tool call format: Tool: [Name] Args: [Args]
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title ✋ Topic 5: Human-in-the-Loop (Safety)
# Concept: Before the agent executes a "Dangerous" tool, it must ask permission.
# This is critical for business automations.

def delete_database():
    return "DATABASE DELETED."

def safe_execute(tool_name):
    # THE SAFETY CHECK
    permission = input(f"⚠️ AGENT WARNING: Making sensitive call to '{tool_name}'. Allow? (y/n): ")
    if permission.lower() == 'y':
        return delete_database()
    else:
        return "Action Denied by User."

# Simulation
print("Agent: I need to clear the old records.")
print(safe_execute("delete_database"))

In [ ]:
# @title 🛒 LAB 2: The Safe Spender
# TASK: Build a logic check.
# If the purchase amount is > $100, ask for human approval.
# If < $100, auto-approve.

def purchase_item(item, price):
    # --- STUDENT WORKSPACE ---
    # Write the if/else logic here using input()
    if price > 100:
        approval = input(f"Approve purchase of {item} for ${price}? (y/n)")
        if approval == 'y': return "Bought."
        else: return "Denied."
    else:
        return "Auto-Bought."
    # -------------------------

# Test it
print(purchase_item("Laptop", 1500))
print(purchase_item("Coffee", 5))

----

### **Phase 3: Memory & Planning**

In [ ]:
# @title 📝 Topic 6: The "Scratchpad" (Long-Term Memory)
# Concept: Agents forget what they did 10 turns ago.
# We give them a "Notepad" to write down important facts that persist.

scratchpad = "FACTS: Door is locked. Key is brass. Pit is south."

prompt = f"""
SCRATCHPAD: {scratchpad}
Action: Jump south.
Critique: Check the scratchpad. Is this safe?
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🗓️ Topic 7: Decomposition (Planning)
# Concept: Agents fail at big tasks.
# We force them to write a Plan BEFORE the first Action.

task = "Host a dinner party for 5 vegans."

prompt = f"""
TASK: {task}
Do not take any actions yet.
First, create a numbered plan of 5 steps to achieve this.
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🏗️ LAB 3: The Project Planner
# TASK: You are the "Brain" of a construction robot.
# Goal: Build a doghouse.
# Write a prompt that forces the AI to list the MATERIALS needed before it starts building.

# --- STUDENT WORKSPACE ---
goal = "Build a wooden doghouse."
planning_prompt = f"""
GOAL: {goal}
Step 1: List all materials needed.
Step 2: List the tools needed.
Step 3: Write the construction steps.
"""
# -------------------------

print(model.generate_content(planning_prompt).text)

----

### **Phase 4: Multi-Agent Systems**

In [ ]:
# @title 🚦 Topic 8: The Router (The Boss)
# Concept: A special agent that just decides WHO should do the work.

team = ["Researcher (Finds facts)", "Writer (Writes poems)", "Coder (Writes Python)"]

prompt = f"""
TEAM: {team}
User: "Find me the stock price of Google."
Who should handle this? Output ONLY the role name.
"""

print(model.generate_content(prompt).text)

In [ ]:
# @title 🤝 Topic 9: The Handoff (Collaboration)
# Concept: Researcher finishes -> Passes output to Writer.

# Step 1: Research
fact = "Mars is red because of iron oxide."
print(f"Researcher found: {fact}")

# Step 2: Handoff to Writer
writer_prompt = f"""
FACT: {fact}
Task: Write a haiku about this fact.
"""
print("\nWriter Output:")
print(model.generate_content(writer_prompt).text)

----

### **Phase 5: NotebookLM (No-Code Agent)**

In [ ]:
# @title 📓 Topic 10: NotebookLM as an Agent (Demo)
# We can't code this, but we can SIMULATE it.
# NotebookLM is effectively a "Research Agent" with a perfect memory of your files.

# INSTRUCTOR:
# 1. Open NotebookLM (notebooklm.google.com).
# 2. Upload a "Company Handbook" PDF.
# 3. Type: "Act as an HR Agent. Based on the handbook, answer: 'How many vacation days do I get?'"

info = """
### Lab 4: The NotebookLM Experiment
**The Task:**
1. Go to NotebookLM.
2. Create a new notebook called "Agent Lab".
3. Upload a complex document (e.g., a car manual or a recipe book).
4. Prompt it: "Act as a Diagnostic Agent. I will describe a sound, and you tell me what is wrong with the car based on the manual."

**Why this is an Agent:**
It observes your input, retrieves knowledge (Memory), and reasons a solution.
"""
display(Markdown(info))

----

# 🏠 Homework: The Customer Support Manager

### The Scenario
You are building the "Brain" for a Customer Support center.
You need a Multi-Agent system to handle incoming emails.

### The Task
Write a Python script that simulates this workflow:
1.  **The Router:** Analyzes an incoming email and decides if it is "Technical Support" or "Billing".
2.  **The Specialist:**
    * If Technical: "Ask for the error code."
    * If Billing: "Ask for the invoice number."
3.  **The Safety Check:** If the user seems angry (detect sentiment), flag for "Human Review".

### Submission
Submit the Python code that chains these 3 prompts together.

In [ ]:
# YOUR CODE GOES HERE